In [ ]:
''' utils.py
Функция draw_boxes(frame, results) — отрисовывает bounding box, метку и confidence (например, "person: 0.92").
Параметры: цвет, толщина линии, шрифт — подобраны так, чтобы не перекрывать сильно исходное изображение.
Может содержать вспомогательные функции для конвертации тензоров в numpy и т.п.
'''
try:
    import cv2
except ImportError:
    raise ImportError('One of required packages is not installed. \nUse: pip install -r requirements.txt')

def draw_boxes( frame, results, class_name, thickness: int = 2, font_scale: float = 0.6, font_thickness: int = 1):
    '''
    Draws bounding boxes and annotations with class name and confidence from YOLO model

    Input:
    - frame (np.ndarray) - original frame in BGR format
    - results (yolo results)
    - class_name (str)
    - thickness (int) = 2 - bounding box thickness
    - font_scale (float) = 0.6
    - font_thickness (int) = 1

    Returns:
    - np.ndarray - annotated frame with bounding boxes
    '''
    annotated_frame = frame.copy()
    boxes = results[0].boxes  # объект Boxes

    # Если нет детекций — возвращаем исходный кадр
    if boxes is None or len(boxes) == 0:
        return annotated_frame

    # Переводим всё в CPU и numpy
    xyxy = boxes.xyxy.cpu().numpy()
    confidences = boxes.conf.cpu().numpy()

    for i in range(len(xyxy)):
        x1, y1, x2, y2 = map(int, xyxy[i])
        conf = confidences[i]

        # рамка
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), thickness)

        # текст
        label = f'{class_name}: {conf:.2f}'
        (text_width, text_height), _ = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)
        cv2.putText(annotated_frame, label, (x1, y1 - 5),
            cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0),
            font_thickness, cv2.LINE_AA)

        # подложка
        overlay = annotated_frame.copy()
        cv2.rectangle(overlay, 
            (x1, y1 - text_height - 10),
            (x1 + text_width, y1),
            (0, 255, 0), -1)  # заливкa
        cv2.addWeighted(overlay, 0.6, annotated_frame, 0.4, 0, annotated_frame)

    return annotated_frame

In [ ]:
''' detector.py
Содержит класс PersonDetector
Загружает предобученную модель
Фильтрует только класс 'person' (или другой заданный)
Возвращает щтфильтрованные результаты
Использует GPU, если есть такая возможность
'''
try:
    import sys
    from ultralytics import YOLO
    from torch.cuda import is_available
except ImportError:
    raise ImportError('One of required packages is not installed. \nUse: pip install -r requirements.txt')

class Detector:
    def __init__(self, model_name = 'yolo11n', object_for_detection = 'person', detection_threshold = 0.2):
        '''
        Initialize detector.
        Detects objects from a certain class (now detects objects from COCO dataset)

        Input: 
        - model_name (str) - path to YOLO model or name of YOLO pretrained model to download
        - object_for_detection (str) - name of class from COCO
        - detection_threshold (float) - from 0 to 1, threshold for model confidence

        Methods: detect - main method; detect objects on an image
        '''
        self.device = 'cuda' if is_available else 'cpu'

        self.model_name = model_name
        try:
            self.model = YOLO(f'{self.model_name}')
            print(f'Loaded {model_name} successfully')
        except Exception as e:
            print( 
                f'Error loading model '{self.model_name}': {e}\n'
                'Possible reasons:\n'
                '  - No internet connection\n'
                '  - Incorrect model name or path to pretrained model (try: yolov11n.pt, yolov11m.pt)\n' 
                )
            sys.exit(1)
        
        if self.device == 'cuda':
            self.model.to(self.device)

        self.object = object_for_detection
        self.obj_id = self.get_obj_id()

        self.threshold = detection_threshold

        print(f'Initialized detector model {self.model_name}, using {self.device} as device')

    def get_obj_id(self):
        '''
        Get id of required object from object name.

        Returns: object id (int)
        '''
        if self.object == 'person': 
            # Default for YOLO
            return 0
        
        id_dict = self.model.names
        if not self.object in id_dict.values():
            raise AttributeError(f'Selected object is not supported by model {self.model_name}')
        
        name_dict = {name:id for id,name in id_dict.items()}
        return name_dict[self.object]

    def detect(self, input_image):
        '''
        Detect object on an image.

        Input: image (np.array)
        Returns: results (YOLO Results - List of objects)
        '''
        input_image = input_image
        results = self.model.predict(input_image, save = False, classes = self.obj_id, conf = self.threshold, verbose = False)
        return results

In [ ]:
''' video_processor.py
Содержит класс VideoProcessor.
Методы:
__init__(detector, output_dir)
process_video(input_path) — читает видео по кадрам через OpenCV, вызывает детектор, отрисовывает результат (через utils.py), сохраняет в выходное видео.
Обеспечивает корректную работу с FPS, разрешением и кодеками на всех ОС.
Создаёт выходной файл вида output_dir/annotated_video.mp4 (or .avi)
'''
try:
    import cv2
    import os
except ImportError:
    raise ImportError('One of required packages is not installed. \nUse: pip install -r requirements.txt')

class VideoProcessor:
    def __init__(self, detector: Detector, output_dir):
        '''
        Processes .mp4 or .avi videos, saves results of detection in output_dir.
        (if output_dir does not exist - creates it)
        DOES NOT save sound of an original video.

        Input: output_dir (str) - existing directory or a directory name to be created

        Methods: process video - main method; process video and write output
        '''
        self.detector = detector

        self.res_path = output_dir
        if not os.path.exists(output_dir):
            os.makedirs(output_dir, exist_ok = False)
            print(f'Created output directory, path: {output_dir}')

    def get_fourcc(self, input_path):
        '''
        Input: input_path (str)

        Returns: fourcc code so that VideoWriter matches encoding of an original video
        also returns string ".mp4" or ".avi"
        '''
        video_type = input_path.split('.')[-1]
        if video_type == 'mp4':
            return '.mp4', cv2.VideoWriter_fourcc(*'XVID')
        elif video_type == 'avi':
            return '.avi', cv2.VideoWriter_fourcc('M', 'J', 'P', 'G')
    
    def make_video_writer(self, video_capture, file_type, fourcc):
        '''
        Creates VideoWriter object with params of an original video.

        Input:
        - video_capture (VideoCapture)
        - file_type (str) - ".mp4" or ".avi"
        - fourcc (cv2.VideoWriter_fourc)

        Returns: cv2.VideoWriter()
        '''

        shapes = [int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
                  int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))]
        fps = video_capture.get(cv2.CAP_PROP_FPS)
        
        if fps <= 0 or shapes[0] <= 0 or shapes[1] <= 0:
            raise ValueError('Incorrect video metadata: FPS or resolution is zero')

        return cv2.VideoWriter(os.path.join(self.res_path, f'output{file_type}'), fourcc, fps, shapes)

    def process_video(self, input_path):
        '''
        Process original video, write annotated video.

        Input: input_path (str)
        '''
        file_type, fourcc = self.get_fourcc(input_path)
        video_capture = cv2.VideoCapture(input_path)
        video_writer = self.make_video_writer(video_capture, file_type, fourcc)

        if not video_capture.isOpened():
            raise IOError(f'Could not open video file {input_path}. Check the codec and file integrity')
        if not video_writer.isOpened():
            raise RuntimeError(f'Failed to create the output video file: {os.path.join(self.res_path, f'output{file_type}')}. Check write permissions')

        print('Starting processing video')
        while video_capture.isOpened():
            read_ok, frame = video_capture.read()

            if not read_ok:
                break

            results = self.detector.detect(frame)
            frame_w_boxes = draw_boxes(frame, results, self.detector.object)

            video_writer.write(frame_w_boxes)

        video_capture.release()
        video_writer.release()
        print(f'Finished detection succesfully, results saved as {os.path.join(self.res_path, f'annotated_video{file_type}')}')

In [ ]:
''' run.py
Главный исполняемый файл.
Парсит аргументы командной строки (input_video_path, output_dir).
Создаёт экземпляры классов из detector.py и video_processor.py.
Запускает обработку видео.
Обрабатывает исключения (некорректные пути, отсутствие файла и т.д.).
'''

import sys
import os

def main():
    if len(sys.argv) > 5 or len(sys.argv) < 3:
        print('Usage: python run.py <path_to_video> <output_directory> <model_name or path_to_model - optional> <detection_threshold - optional>')
        sys.exit(1)

    input_path = sys.argv[1]
    out_path = sys.argv[2]

    # check if input_path is a path to a video file
    if not os.path.isfile(input_path):
        print(f'{input_path} has no video file')
        sys.exit(1)
    if not input_path.lower().endswith(('.mp4', '.avi')):
        print('Check file format. input_path must be one of .mp4 or .avi')
        sys.exit(1)

    #initialise Detector with params from argv or with default params
    if len(sys.argv) == 3:
        #init Detector with default params
        detector = Detector()
    elif len(sys.argv) == 4:
        try:
            detection_threshold = float(sys.argv[3])
            model_name = 'yolo11n'
        except ValueError:
            detection_threshold = 0.2
            model_name = sys.argv[3]
        detector = Detector(model_name = model_name, detection_threshold=detection_threshold)
    else:
        try:
            detection_threshold = float(sys.argv[4])
            model_name = sys.argv[3]
            detector = Detector(model_name = model_name, detection_threshold=detection_threshold)
        except ValueError:
            print('Usage: python run.py <path_to_video> <output_directory> <model_name or path_to_model - optional> <detection_threshold - optional>')
            sys.exit(1)

    
    processor = VideoProcessor(detector, out_path)
    processor.process_video(input_path)

Initialized detector model yolo11n, using cuda as device
Finished detection succesfully, results saved as results_1\output.mp4
